# Machine Learning Jump-Start Workbook

Let's build a basic machine learning model. First thing's first; we need to
import the packages we need - specifically, PyTorch.
Documentation can be found at https://docs.pytorch.org/docs/stable/index.html

### Install relevant python packages via pip (only uncomment and run if these are necessary)

In [ ]:
# !python -m pip install torch torchvision tqdm

In [ ]:
import torch

Using PyTorch, we can generate practically any model we would like to. PyTorch is very powerful, as you have full control over every component of the network, while wrapping together many useful components.

To start, we can make a very simple 2-layer network.

In [ ]:
linear_layer = torch.nn.Linear(28*28, 100, bias=True)

In [ ]:
linear_layer.weight.shape

torch.Size([100, 784])

In [ ]:
class SimpleModel(torch.nn.Module):
  def __init__(self, input_size, output_size):
    super(SimpleModel, self).__init__()
    self.layer1 = torch.nn.Linear(input_size, 100, bias=True)
    self.layer2 = torch.nn.Linear(100, output_size, bias=True)
    self.activation = torch.nn.ReLU()

In [ ]:
model = SimpleModel(28*28, 10)
print(model)

SimpleModel(
  (layer1): Linear(in_features=784, out_features=100, bias=True)
  (layer2): Linear(in_features=100, out_features=10, bias=True)
  (activation): ReLU()
)


In [ ]:
model.layer1.weight.shape

torch.Size([100, 784])

**Intermission**

In [ ]:
from torchvision.datasets import MNIST

In [ ]:
from torchvision.transforms import ToTensor
train_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 54.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.71MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.75MB/s]


In [ ]:
sample = next(iter(train_dataloader))
sample[0].shape
sample_t = sample[0].view(28*28, -1)
sample_t.shape

torch.Size([784, 128])

In [ ]:
class SimpleModel(torch.nn.Module):
  def __init__(self, input_size, output_size):
    super(SimpleModel, self).__init__()
    self.layer1 = torch.nn.Linear(input_size, 100, bias=True)
    self.layer2 = torch.nn.Linear(100, output_size, bias=True)
    self.activation = torch.nn.ReLU()

  def forward(self, X):
    y = self.layer1(X)
    y = self.activation(y)
    y = self.layer2(y)
    y = self.activation(y)
    return y

model = SimpleModel(28*28, 10)

Training loop

In [ ]:
from tqdm import tqdm
# optim = torch.optim.Adam(lr=10e-3)
optim = torch.optim.SGD(model.parameters(), lr=10e-3)
loss_fn = torch.nn.CrossEntropyLoss()
n_epochs = 2

for epoch_i in range(n_epochs):
  print(f'Epoch {epoch_i+1}/{n_epochs}')
  for batch_i, (X,y) in enumerate(tqdm(train_dataloader)):
    X = X.view(-1, 28*28)
    output = model(X)
    loss = loss_fn(output, y)
    loss.backward()
    optim.step()
    optim.zero_grad()

Epoch 1/2


100%|██████████| 469/469 [00:09<00:00, 47.61it/s]


Epoch 2/2


100%|██████████| 469/469 [00:10<00:00, 46.07it/s]


Now let's test to see how good our model has become!

In [ ]:
model.eval()
test_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=True)

(test_X, test_y) = next(iter(test_dataloader))
test_X = test_X.view(-1, 28*28)

output = model(test_X)
predictions = torch.argmax(output, dim=1)
print(predictions)
print(test_y)
corrects = torch.where(predictions == test_y, 1, 0)
print(f'Number of correct predictions: {sum(corrects)}/{len(corrects)}')

tensor([0, 1, 9, 1, 9, 3, 0, 0, 3, 7, 6, 3, 3, 6, 4, 6, 9, 1, 2, 3, 4, 3, 9, 4,
        2, 9, 9, 9, 2, 3, 4, 9, 1, 9, 0, 5, 1, 7, 7, 9, 6, 7, 5, 0, 5, 3, 2, 0,
        4, 3, 4, 5, 3, 0, 1, 6, 0, 4, 3, 7, 6, 9, 0, 7, 4, 1, 1, 6, 5, 7, 9, 3,
        2, 7, 1, 0, 2, 0, 7, 5, 0, 0, 6, 9, 1, 2, 5, 9, 0, 6, 1, 4, 5, 7, 1, 9,
        0, 5, 4, 0, 9, 9, 7, 1, 5, 5, 4, 3, 1, 6, 1, 4, 3, 2, 1, 4, 6, 3, 9, 4,
        1, 3, 5, 9, 6, 1, 7, 3])
tensor([0, 1, 8, 1, 4, 3, 0, 0, 5, 7, 6, 8, 9, 6, 9, 8, 9, 1, 2, 8, 8, 3, 8, 4,
        2, 8, 0, 9, 8, 8, 9, 9, 1, 9, 9, 8, 1, 2, 7, 9, 6, 7, 5, 0, 0, 5, 7, 0,
        4, 5, 4, 5, 3, 0, 1, 6, 0, 9, 3, 7, 6, 9, 0, 7, 4, 1, 1, 6, 4, 7, 8, 3,
        2, 7, 1, 0, 2, 0, 7, 5, 0, 0, 6, 7, 1, 2, 5, 9, 0, 6, 1, 4, 5, 7, 8, 9,
        7, 5, 4, 0, 9, 9, 5, 2, 5, 5, 5, 3, 1, 6, 1, 4, 3, 2, 7, 7, 6, 3, 9, 5,
        1, 3, 5, 5, 6, 1, 5, 3])
Number of correct predictions: 92/128


# Putting everything together: Validation/Training Loop

In [ ]:
import torch

class SimpleModel(torch.nn.Module):
  def __init__(self, input_size, output_size):
    super(SimpleModel, self).__init__()
    self.layer1 = torch.nn.Linear(input_size, 100, bias=True)
    self.layer2 = torch.nn.Linear(100, output_size, bias=True)
    self.activation = torch.nn.ReLU()

  def forward(self, X):
    y = self.layer1(X)
    y = self.activation(y)
    y = self.layer2(y)
    y = self.activation(y)
    return y

model = SimpleModel(28*28, 10)

In [ ]:
from tqdm import tqdm

def train_loop():
  model.train()
  corrects = []
  for batch_i, (X,y) in enumerate(tqdm(train_dataloader)):
    X = X.view(-1, 28*28)
    output = model(X)
    loss = loss_fn(output, y)
    loss.backward()
    optim.step()
    optim.zero_grad()

    predictions_i = torch.argmax(output, dim=1)
    corrects_i = torch.where(predictions_i == y, 1, 0)
    corrects.extend(corrects_i)
  print(f'Number of correct predictions: {sum(corrects)}/{len(corrects)}')
  print(f'Accuracy on training dataset: {sum(corrects)/len(corrects)*100:.3f}%')

def test_loop():
  model.eval()
  corrects = []
  for batch_i, (X,y) in enumerate(tqdm(train_dataloader)):
    X = X.view(-1, 28*28)
    output = model(X)

    predictions_i = torch.argmax(output, dim=1)
    corrects_i = torch.where(predictions_i == y, 1, 0)
    corrects.extend(corrects_i)
  print(f'Number of correct predictions: {sum(corrects)}/{len(corrects)}')
  print(f'Accuracy on test dataset: {sum(corrects)/len(corrects)*100:.3f}%')

In [ ]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
train_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataset = MNIST(train=True, root='data/', download=True, transform=ToTensor())
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=True)

optim = torch.optim.SGD(model.parameters(), lr=10e-3)
loss_fn = torch.nn.CrossEntropyLoss()
n_epochs = 100

print('Testing model before training')
test_loop()
for epoch_i in range(n_epochs):
  print(f'\nEpoch {epoch_i+1}/{n_epochs}')
  print('Training model...')
  train_loop()
  print('Testing model...')
  test_loop()

Testing model before training


100%|██████████| 469/469 [00:08<00:00, 53.18it/s]


Number of correct predictions: 8013/60000
Accuracy on test dataset: 13.355%

Epoch 1/100
Training model...


100%|██████████| 469/469 [00:09<00:00, 50.92it/s]


Number of correct predictions: 31637/60000
Accuracy on training dataset: 52.728%
Testing model...


...

Number of correct predictions: 51716/60000
Accuracy on test dataset: 86.193%

Epoch 100/100
Training model...


100%|██████████| 469/469 [00:09<00:00, 49.18it/s]


Number of correct predictions: 51721/60000
Accuracy on training dataset: 86.202%
Testing model...


100%|██████████| 469/469 [00:08<00:00, 53.55it/s]


Number of correct predictions: 51725/60000
Accuracy on test dataset: 86.208%
